# ddml_nsw

> 对应代码：`EconometricsCode/code_in_notes/Chap.12/ddml_nsw.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.12`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.12")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空内存并关闭分屏，然后载入 NSW 实验数据集（LaLonde 1986 的经典数据，处理组来自真实随机实验）；被注释掉的一行原本是合并 CPS 对照组，这里只保留实验样本。

In [ ]:
%%stata
clear
set more off
use ../datasets/Lalonde_nsw/nsw.dta
// append using ../datasets/Lalonde_nsw/cps_controls.dta

把 age 到 nodegree 这一列连续的协变量存入局部宏 controls，作为混淆变量 X，后面两处机器学习方程都要用到它。

In [ ]:
%%stata
local controls "age-nodegree"

初始化 ddml（Double/Debiased Machine Learning）流程，指定 interactive 模式——即部分线性/交互式模型，需要分别学习结果方程 E[Y|X,D] 与倾向得分 E[D|X]；kfolds(10) 表示把样本随机分成 10 折做交叉拟合（cross-fitting），用样本外预测来构造正交化得分，从而避免用同一份数据既拟合又估计带来的过拟合偏误。

In [ ]:
%%stata
ddml init interactive, kfolds(10)

设定结果方程 E[Y|X,D] 的学习器：因变量是 1978 年真实收入 re78，用 pystacked 调用 Python 的 scikit-learn，以随机森林（rf）做回归，树的数量设为 200。这一步得到条件期望函数的灵活拟合，用机器学习捕捉 X 与 Y 之间的非线性关系。

In [ ]:
%%stata
ddml E[Y|X,D]: pystacked re78  `controls', type(regress) method(rf) cmdopt1(n_estimators(200))

设定处理方程 E[D|X]（即倾向得分）的学习器：处理变量是 treat，同样用随机森林但改为分类（class），因为 D 是 0-1 变量。两个方程共同构成 Neyman 正交得分所需的“冗余参数”（nuisance parameters）。

In [ ]:
%%stata
ddml E[D|X]: pystacked treat  `controls', type(class) method(rf) cmdopt1(n_estimators(200))

执行交叉拟合：在 10 折样本上轮流用 9 折训练两个机器学习模型、在剩余 1 折上预测，从而每个观测的拟合值都来自未包含它的模型，得到干净的样本外预测残差。

In [ ]:
%%stata
ddml crossfit

基于交叉拟合得到的残差，用 Robinson 变换后的正交化矩条件估计平均处理效应；robust 选项报告异方差稳健标准误。由于 DML 的正交性，即使两个机器学习方程收敛速度较慢，处理效应估计量仍以根号 n 速度渐近正态，可以做常规推断。

In [ ]:
%%stata
ddml estimate, robust